In [ ]:
from pathlib import Path
import os

def _analysis_data_root():
    if os.environ.get("ANALYSIS_DATA_DIR"):
        return Path(os.environ["ANALYSIS_DATA_DIR"]).expanduser().resolve()
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / "analysis_data", base / "A_paper" / "analysis_data"):
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set ANALYSIS_DATA_DIR to the extracted analysis_data directory")

DATA_ROOT = _analysis_data_root()
RESULTS_ROOT = Path(os.environ.get("ANALYSIS_RESULTS_DIR", DATA_ROOT.parent / "analysis_code" / "results")).expanduser().resolve()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Batch-convert several RNA gene-rank CSVs for one species (Homo sapiens)
into Macaca mulatta ortholog gene CSVs.

Only CSVs whose file names satisfy both conditions are processed:
    1. contain "homo_sapiens" (case-insensitive)
    2. contain "_rna_" (case-insensitive)

Macaca ATAC files, scripts and any other files in the directory are therefore
left untouched.

Input CSV format:
    gene,score

Also accepted:
    gene_id,score
    gene_id,gene_symbol,score

Default input directory:
    analysis_data/Figure6/brain/rankings

Default output directory (created automatically):
    <input directory>/human_rna_to_macaca

For each input sample the script writes:
    1. converted/<original name with homo_sapiens replaced by Macaca_mulatta>
       the final macaque gene table: gene,gene_symbol,score

    2. mapping/<input stem>__full_mapping.csv
       every candidate ortholog relationship

    3. mapping/<input stem>__best_mapping.csv
       the best ortholog chosen for each human gene

    4. unmapped/<input stem>__unmapped.csv
       human genes that could not be mapped

One extra batch-level output:
    batch_conversion_summary.csv

Note:
    The output score is inherited directly from the corresponding human RNA
    input file; it is not recomputed on the macaque data.
"""

from __future__ import annotations

import sys
import traceback
from pathlib import Path

import pandas as pd


INPUT_DIR = DATA_ROOT / "Figure6/brain/rankings"

HOMOLOG_FILE = DATA_ROOT / "Figure6/brain/human_macaque_homology.csv"

OUTPUT_DIR = RESULTS_ROOT / "Figure6/brain/human_rna_to_macaca"

KEEP_ONLY_BEST_HOMOLOG = True
PREFER_ONE_TO_ONE = True


# ============================================================
# Shared helpers
# ============================================================
def clean_gene_id(value: object) -> str:
    """Clean an Ensembl ID and strip its version suffix."""
    if pd.isna(value):
        return ""

    value = str(value).strip()
    if value.lower() in {"", "nan", "none", "na", "<na>"}:
        return ""

    return value.split(".", maxsplit=1)[0]


def clean_text(value: object) -> str:
    """Clean a plain text value."""
    if pd.isna(value):
        return ""

    value = str(value).strip()
    if value.lower() in {"", "nan", "none", "na", "<na>"}:
        return ""

    return value


def orthology_priority(value: object) -> int:
    """Ortholog priority; higher values win."""
    priority_map = {
        "ortholog_one2one": 4,
        "ortholog_one2many": 3,
        "ortholog_many2one": 3,
        "ortholog_many2many": 2,
    }
    return priority_map.get(clean_text(value), 1)


def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")


def require_directory(path: Path) -> None:
    if not path.is_dir():
        raise NotADirectoryError(f"Directory not found: {path}")


# ============================================================
# Input-file discovery
# ============================================================
def is_target_human_rna_csv(path: Path) -> bool:
    """Recognise human RNA CSVs only; explicitly exclude ATAC and other species."""
    name = path.name.lower()
    return (
        path.is_file()
        and path.suffix.lower() == ".csv"
        and "homo_sapiens" in name
        and "_rna_" in name
        and "_atac_" not in name
    )


def discover_input_files(input_dir: Path) -> list[Path]:
    """Find every human RNA CSV under the input directory without recursing into the output directory."""
    require_directory(input_dir)

    # Match the current file-naming pattern directly:
    # mosp_atac_ntv3_pre_homo_sapiens_<sample>_rna_<celltype>.csv
    candidates = list(input_dir.glob("*homo_sapiens*_rna_*.csv"))

    # Then apply a case-insensitive fallback filter to absorb naming-case differences.
    if not candidates:
        candidates = [
            path
            for path in input_dir.iterdir()
            if is_target_human_rna_csv(path)
        ]

    return sorted(
        (path for path in candidates if path.is_file()),
        key=lambda p: p.name.lower(),
    )


def build_output_name(input_file: Path) -> str:
    """Replace the human species name in a filename with the macaque species name."""
    name = input_file.name

    for token in ("homo_sapiens", "Homo_sapiens", "HOMO_SAPIENS"):
        if token in name:
            return name.replace(token, "Macaca_mulatta", 1)

    # Normally unreachable because discover() has already filtered.
    return f"{input_file.stem}_human2macaca.csv"


# ============================================================
# Read the input gene table
# ============================================================
def load_human_genes(path: Path) -> pd.DataFrame:
    """
    Read a human RNA ranking table.

    Recommended input format:
        gene,score

    where gene is a human Ensembl gene ID. For compatibility with older files,
    gene_id is also accepted; gene_symbol is optional. When gene_symbol is
    absent it is filled in later from Human.gene.name in the BioMart ortholog
    table.
    """
    require_file(path)

    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed:")].copy()
    df.columns = df.columns.astype(str).str.strip()

    # Accept both the current gene,score and the older gene_id,... format.
    if "gene" in df.columns:
        gene_column = "gene"
    elif "gene_id" in df.columns:
        gene_column = "gene_id"
    else:
        raise ValueError(
            "Input must contain a 'gene' or 'gene_id' column; "
            f"observed columns: {df.columns.tolist()}"
        )

    if "score" not in df.columns:
        raise ValueError(
            "Input missing column: score; "
            f"observed columns: {df.columns.tolist()}"
        )

    selected_columns = [gene_column, "score"]
    if "gene_symbol" in df.columns:
        selected_columns.insert(1, "gene_symbol")

    df = df[selected_columns].copy()
    df = df.rename(columns={gene_column: "gene_id"})

    if "gene_symbol" not in df.columns:
        df["gene_symbol"] = ""

    df = df[["gene_id", "gene_symbol", "score"]].copy()
    df["gene_id"] = df["gene_id"].map(clean_gene_id)
    df["gene_symbol"] = df["gene_symbol"].map(clean_text)
    df["score"] = pd.to_numeric(df["score"], errors="coerce")

    invalid_id = df["gene_id"].eq("")
    invalid_score = df["score"].isna()

    if invalid_id.any():
        raise ValueError(
            "Input contains empty/invalid gene rows:\n"
            + df.loc[invalid_id].to_string(index=False)
        )

    if invalid_score.any():
        raise ValueError(
            "Input contains invalid score rows:\n"
            + df.loc[invalid_score].to_string(index=False)
        )

    if df["gene_id"].duplicated().any():
        duplicated = df.loc[df["gene_id"].duplicated(keep=False)]
        raise ValueError(
            "Input contains duplicated gene IDs:\n"
            + duplicated.to_string(index=False)
        )

    df.insert(0, "human_rank", range(1, len(df) + 1))
    return df


# ============================================================
# Read the BioMart ortholog table (once per batch)
# ============================================================
def load_homolog_table(path: Path) -> pd.DataFrame:
    require_file(path)

    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    df.columns = df.columns.astype(str).str.strip()

    required = {
        "Gene.stable.ID",
        "Gene.name",
        "Human.gene.stable.ID",
        "Human.gene.name",
        "homolog_orthology_type",
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Homolog table missing columns: {sorted(missing)}; "
            f"observed columns: {df.columns.tolist()}"
        )

    df = df[
        [
            "Gene.stable.ID",
            "Gene.name",
            "Human.gene.stable.ID",
            "Human.gene.name",
            "homolog_orthology_type",
        ]
    ].rename(
        columns={
            "Gene.stable.ID": "macaca_gene_id",
            "Gene.name": "macaca_gene_symbol",
            "Human.gene.stable.ID": "human_gene_id",
            "Human.gene.name": "human_biomart_symbol",
            "homolog_orthology_type": "orthology_type",
        }
    )

    df["macaca_gene_id"] = df["macaca_gene_id"].map(clean_gene_id)
    df["macaca_gene_symbol"] = df["macaca_gene_symbol"].map(clean_text)
    df["human_gene_id"] = df["human_gene_id"].map(clean_gene_id)
    df["human_biomart_symbol"] = df["human_biomart_symbol"].map(clean_text)
    df["orthology_type"] = df["orthology_type"].map(clean_text)

    df = df.loc[
        df["macaca_gene_id"].ne("") & df["human_gene_id"].ne("")
    ].copy()

    df = df.drop_duplicates(
        subset=["human_gene_id", "macaca_gene_id", "orthology_type"]
    ).reset_index(drop=True)

    df["orthology_priority"] = df["orthology_type"].map(orthology_priority)
    df["has_macaca_symbol"] = df["macaca_gene_symbol"].ne("").astype(int)

    return df


# ============================================================
# Mapping and best-ortholog selection
# ============================================================
def build_mapping(
    human_df: pd.DataFrame,
    homolog_df: pd.DataFrame,
) -> pd.DataFrame:
    human_for_merge = human_df.rename(
        columns={
            "gene_id": "human_gene_id",
            "gene_symbol": "human_gene_symbol",
        }
    )

    relevant = homolog_df.loc[
        homolog_df["human_gene_id"].isin(set(human_df["gene_id"]))
    ].copy()

    mapping_df = human_for_merge.merge(
        relevant,
        on="human_gene_id",
        how="left",
        validate="one_to_many",
    )

    for column in [
        "human_gene_symbol",
        "human_biomart_symbol",
        "macaca_gene_id",
        "macaca_gene_symbol",
        "orthology_type",
    ]:
        mapping_df[column] = mapping_df[column].fillna("").map(clean_text)

    # The current input usually has only gene,score, so prefer the input symbol;
    # when the input symbol is empty, fill it from Human.gene.name in BioMart.
    missing_human_symbol = mapping_df["human_gene_symbol"].eq("")
    mapping_df.loc[missing_human_symbol, "human_gene_symbol"] = (
        mapping_df.loc[missing_human_symbol, "human_biomart_symbol"]
    )

    mapping_df["orthology_priority"] = pd.to_numeric(
        mapping_df["orthology_priority"], errors="coerce"
    ).fillna(0).astype(int)

    mapping_df["has_macaca_symbol"] = pd.to_numeric(
        mapping_df["has_macaca_symbol"], errors="coerce"
    ).fillna(0).astype(int)

    mapping_df["has_macaca_mapping"] = mapping_df["macaca_gene_id"].ne("").astype(int)
    return mapping_df


def choose_best_homolog(mapping_df: pd.DataFrame) -> pd.DataFrame:
    ranked = mapping_df.copy()

    if not PREFER_ONE_TO_ONE:
        ranked["orthology_priority"] = ranked["has_macaca_mapping"]

    ranked = ranked.sort_values(
        by=[
            "human_rank",
            "has_macaca_mapping",
            "orthology_priority",
            "has_macaca_symbol",
            "macaca_gene_id",
        ],
        ascending=[True, False, False, False, True],
        kind="stable",
    )

    return (
        ranked.groupby("human_rank", sort=False, as_index=False)
        .head(1)
        .sort_values("human_rank")
        .reset_index(drop=True)
    )


def build_macaca_output(
    mapping_df: pd.DataFrame,
    best_df: pd.DataFrame,
) -> pd.DataFrame:
    selected = best_df.copy() if KEEP_ONLY_BEST_HOMOLOG else mapping_df.copy()
    selected = selected.loc[selected["macaca_gene_id"].ne("")].copy()

    selected = selected.sort_values(
        ["human_rank", "orthology_priority", "macaca_gene_id"],
        ascending=[True, False, True],
        kind="stable",
    )

    macaca_df = selected.rename(
        columns={
            "macaca_gene_id": "gene",
            "macaca_gene_symbol": "gene_symbol",
        }
    )[["gene", "gene_symbol", "score"]].copy()

    # Several human genes may map to the same macaque gene.
    # Keep the entry whose source human rank is best.
    macaca_df = macaca_df.drop_duplicates(subset=["gene"], keep="first")
    return macaca_df.reset_index(drop=True)


# ============================================================
# Per-sample outputs
# ============================================================
def process_one_file(
    input_file: Path,
    homolog_df: pd.DataFrame,
    converted_dir: Path,
    mapping_dir: Path,
    unmapped_dir: Path,
) -> dict[str, object]:
    human_df = load_human_genes(input_file)
    mapping_df = build_mapping(human_df, homolog_df)
    best_df = choose_best_homolog(mapping_df)
    macaca_df = build_macaca_output(mapping_df, best_df)

    output_name = build_output_name(input_file)
    converted_file = converted_dir / output_name
    full_mapping_file = mapping_dir / f"{input_file.stem}__full_mapping.csv"
    best_mapping_file = mapping_dir / f"{input_file.stem}__best_mapping.csv"
    unmapped_file = unmapped_dir / f"{input_file.stem}__unmapped.csv"

    full_columns = [
        "human_rank",
        "human_gene_id",
        "human_gene_symbol",
        "score",
        "human_biomart_symbol",
        "macaca_gene_id",
        "macaca_gene_symbol",
        "orthology_type",
        "orthology_priority",
    ]
    mapping_df[full_columns].to_csv(full_mapping_file, index=False)

    best_columns = [
        "human_rank",
        "human_gene_id",
        "human_gene_symbol",
        "score",
        "human_biomart_symbol",
        "macaca_gene_id",
        "macaca_gene_symbol",
        "orthology_type",
    ]
    best_df[best_columns].to_csv(best_mapping_file, index=False)

    unmapped_df = best_df.loc[
        best_df["macaca_gene_id"].eq(""),
        ["human_rank", "human_gene_id", "human_gene_symbol", "score"],
    ].copy()
    unmapped_df.to_csv(unmapped_file, index=False)

    macaca_df.to_csv(converted_file, index=False)

    mapped_count = int(best_df["macaca_gene_id"].ne("").sum())
    one2one_count = int(best_df["orthology_type"].eq("ortholog_one2one").sum())
    multi_homolog_count = int(
        mapping_df.loc[mapping_df["macaca_gene_id"].ne("")]
        .groupby("human_gene_id")["macaca_gene_id"]
        .nunique()
        .gt(1)
        .sum()
    )

    return {
        "input_file": str(input_file),
        "output_file": str(converted_file),
        "status": "SUCCESS",
        "input_human_genes": len(human_df),
        "mapped_human_genes": mapped_count,
        "selected_one2one": one2one_count,
        "human_genes_with_multiple_homologs": multi_homolog_count,
        "final_macaca_genes": len(macaca_df),
        "unmapped_human_genes": len(unmapped_df),
        "error": "",
    }


# ============================================================
# Main workflow
# ============================================================
def main() -> None:
    input_dir = INPUT_DIR.resolve()
    homolog_file = HOMOLOG_FILE.resolve()
    output_dir = OUTPUT_DIR.resolve()

    require_directory(input_dir)
    require_file(homolog_file)

    if output_dir == input_dir:
        raise ValueError("OUTPUT_DIR must differ from INPUT_DIR")

    converted_dir = output_dir / "converted"
    mapping_dir = output_dir / "mapping"
    unmapped_dir = output_dir / "unmapped"

    for directory in (output_dir, converted_dir, mapping_dir, unmapped_dir):
        directory.mkdir(parents=True, exist_ok=True)

    input_files = discover_input_files(input_dir)
    if not input_files:
        visible_entries = sorted(path.name for path in input_dir.iterdir())
        print("\nWhat Python currently sees in INPUT_DIR:", file=sys.stderr)
        if visible_entries:
            for name in visible_entries:
                print(f"  - {name!r}", file=sys.stderr)
        else:
            print("  <directory is empty>", file=sys.stderr)

        raise FileNotFoundError(
            "No human RNA CSV found. The actual match pattern is "
            "'*homo_sapiens*_rna_*.csv'. Check that the current Jupyter kernel can access that mounted directory "
            "and that you are running the latest script/cell."
        )

    print("=" * 88)
    print("Batch human RNA -> Macaca ortholog conversion")
    print(f"Input directory : {input_dir}")
    print(f"Homolog table   : {homolog_file}")
    print(f"Output directory: {output_dir}")
    print(f"Matched files   : {len(input_files)}")
    print("=" * 88)

    for path in input_files:
        print(f"  - {path.name}")

    print("\nLoading homolog table once for the whole batch...")
    homolog_df = load_homolog_table(homolog_file)
    print(f"Valid homolog records: {len(homolog_df):,}\n")

    summary_rows: list[dict[str, object]] = []

    for index, input_file in enumerate(input_files, start=1):
        print(f"[{index}/{len(input_files)}] Processing {input_file.name}")

        try:
            result = process_one_file(
                input_file=input_file,
                homolog_df=homolog_df,
                converted_dir=converted_dir,
                mapping_dir=mapping_dir,
                unmapped_dir=unmapped_dir,
            )
            summary_rows.append(result)
            print(
                "    SUCCESS | "
                f"input={result['input_human_genes']} | "
                f"mapped={result['mapped_human_genes']} | "
                f"final={result['final_macaca_genes']} | "
                f"unmapped={result['unmapped_human_genes']}"
            )
            print(f"    output: {result['output_file']}")

        except Exception as exc:
            summary_rows.append(
                {
                    "input_file": str(input_file),
                    "output_file": "",
                    "status": "FAILED",
                    "input_human_genes": "",
                    "mapped_human_genes": "",
                    "selected_one2one": "",
                    "human_genes_with_multiple_homologs": "",
                    "final_macaca_genes": "",
                    "unmapped_human_genes": "",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            print(f"    FAILED | {type(exc).__name__}: {exc}", file=sys.stderr)
            traceback.print_exc()

    summary_df = pd.DataFrame(summary_rows)
    summary_file = output_dir / "batch_conversion_summary.csv"
    summary_df.to_csv(summary_file, index=False)

    success_count = int(summary_df["status"].eq("SUCCESS").sum())
    failed_count = int(summary_df["status"].eq("FAILED").sum())

    print("\n" + "=" * 88)
    print(f"Finished: total={len(summary_df)}, success={success_count}, failed={failed_count}")
    print(f"Converted files : {converted_dir}")
    print(f"Mapping files   : {mapping_dir}")
    print(f"Unmapped files  : {unmapped_dir}")
    print(f"Batch summary   : {summary_file}")
    print("=" * 88)

    if failed_count > 0:
        raise RuntimeError(
            f"{failed_count} files failed in the batch; see batch_conversion_summary.csv"
        )


if __name__ == "__main__":
    # Do not use sys.exit(), which triggers a second traceback in Jupyter/IPython.
    main()
